In [1]:
import pickle
import pandas as pd
import os
from env_action.environment  import FJSP_under_uncertainties_Env
from stable_baselines3       import DQN

models_dir = 'PR-DDQN_tight_duedate_JA_only_trainfreq100step_2024-12-17_05-59-19'
model_path = os.path.join('models', models_dir, f"DQN_.zip")

setting             = 'TIGHT_DUEDATE'
directory           = f'DATA/{setting}_VALIDATION'
planning_horizon    = 480*60
critical_machines   = {5, 6, 7, 8, 9, 10, 11, 12, 13, 21, 22, 26, 27}
ReworkProbability   = 0.03
maxtime             = 20
PopSize             = 40
WeibullDistribution = pd.read_excel('DATA/DataMaster.xlsx', sheet_name='Distribution')
K                   = 30
maxJob              = 1320 # for normalization
maxOpe              = 5760 # for normalization

tight_duedate_setting   = True if "tight_duedate" in models_dir else False
JA_only_setting         = True if "JA_only" in models_dir else False
with open('DATA/ComponentMaster.pkl', 'rb') as f:
        master = pickle.load(f)

In [2]:
with open(f'{directory}/pickle_valid_instances_480.pkl', 'rb') as f:
    valid_instances = pickle.load(f)
with open(f'{directory}/pickle_valid_scenarios_480.pkl', 'rb') as f:
    valid_scenarios = pickle.load(f)

results      = []
method       = 'PR-DDQN'
InstanceList = [f'valid{i+1}' for i in range(10)]
ScenarioList = ['A', 'B', 'C']
reward_ratio = 0.99

valenv = FJSP_under_uncertainties_Env(False, False, valid_instances, valid_scenarios, K, WeibullDistribution, critical_machines, 
                                      ReworkProbability, planning_horizon, PopSize, maxtime, maxJob, maxOpe, reward_ratio,
                                      master, tight_duedate_setting, JA_only_setting)


model = DQN.load(model_path, env=valenv)

for run_time in range(1):
    print("----------- Run Time", run_time)
    for instance_id in InstanceList:
        print("-----------", instance_id)
        for scenario_id in ScenarioList:
            print("-----", scenario_id)
            # Reset the environment with the new dataset
            # valenv.reset(test=True, 
            #         datatest=instance_id, 
            #         scenariotest=scenario_id)
            
            obs, info = valenv.reset(test=True, 
                    datatest=instance_id, 
                    scenariotest=scenario_id)
            done = False
            
            while not done:
                action, _states = model.predict(obs, deterministic= True)
                obs, reward, done, truncated, info = valenv.step(action)
            
            tardiness = valenv.calc_tardiness()
        
            results.append({'RunTime'   : run_time,
                            'Method'    : method,
                            'InstanceID': instance_id,
                            'ScenarioID': scenario_id,
                            'Tardiness' : tardiness
                            })


Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
----------- Run Time 0
----------- valid1
----- A
DONEEEEEEEEEEEEEEEEEE
1 0 1 ---- 0 0 0.0
Method selection:                    RCRS
1 0 1 ---- 0 0 0.0
Method selection:                    CDR2
1 0 1 ---- 0 0 0.0
Method selection:                    RCRS
Breaking ----------------- 5 5 5
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    TS
1 0 1 ---- 0 0 0.0
Method selection:                    CDR2
1 0 1 ---- 0 0 0.0
Method selection:                    RCRS
1 0 1 ---- 0 0 0.0
Method selection:                    CDR2
1 0 1 ---- 0 0 0.0
Method selection:                    CDR2
1 0 1 ---- 0 0 0.0
Method selection: 

In [3]:
df = pd.DataFrame(results)
file_name = f"VALIDATION/{models_dir}_1to10.xlsx"
df.to_excel(file_name, index=False)